# Configure Route 53 failover routing across two regional ALBs and observe automatic failover when one region goes dark

A customer SLA in your team's master service agreement reads "site available even during a full AWS Region outage." The founding engineer asked you to pick a Route 53 routing policy and prove it works. You have one session to stand up two regional ALBs, wire failover records with health checks, prove the primary serves traffic while it is healthy, then break the primary's health check and confirm Route 53 flips DNS to the secondary inside the SLA's 3-minute window.

You will build:

- An ALB + EC2 instance in **us-east-1** (Architecture A, primary region) serving an HTTP response body that contains the substring `us-east-1`.
- An ALB + EC2 instance in **us-west-2** (Architecture B, secondary region) with the same shape, body says `us-west-2`.
- A Route 53 PUBLIC hosted zone for a lab-only domain (`labrun-{account-id}.example`), tagged for cleanup.
- Two Route 53 health checks (one per regional ALB) probing the public DNS.
- Two failover routing records (`Failover=PRIMARY` and `Failover=SECONDARY`) on the same record name, each with a HealthCheckId pointing at its regional health check.

Then you break the primary's health check (by re-pointing the us-east-1 target group to a non-existent path), wait for the Route 53 health-check evaluator to mark it failed, and call `test_dns_answer` to confirm the DNS response now returns the us-west-2 ALB.

**Time.** Plan for about 65 minutes hands-on. The two longest waits are the regional instance launches in Task 1 (3-5 minutes each, in parallel) and the failover propagation in Task 5 (~2 minutes after the health-check break).

**Cost.** This is the second-highest-cost lab in the SAA track after Lab 3. Two ALBs at $0.0225/hour combined are about 4.5 cents per hour. Two t4g.nano at sub-cent rates. Route 53 hosted zone at $0.50/month prorated to fractions of a cent. Two health checks at $0.50/check/month after the first 50 (free tier). Session range: $0.10 to $0.30. The cleanup cell tears down both regions; the multi-region orphan scan catches anything that escapes.

This lab maps to AWS SAA-C03 Domain 2 (Design Resilient Architectures, 26%).

In [ ]:
# NBVAL_SKIP
# Install labrun-checks. Pinned per LAB-CREATION-BLUEPRINT.md build rules.

!pip install --quiet labrun-checks==0.6.0

In [ ]:
# NBVAL_SKIP
# Imports and per-lab constants. This is a multi-region lab; every boto3
# client is constructed per region so the calls land in the right place.

import atexit
import base64
import datetime as dt
import getpass
import json
import socket
import time
from urllib.error import URLError
from urllib.request import urlopen

import boto3
from botocore.exceptions import ClientError
from IPython.display import clear_output

from labrun_checks import (
    register,
    check,
    cleanup,
    run_cleanup,
    CleanupResource,
)

LAB_ID = "route53-failover-routing"
LAB_TAG_KEY = "labrun:lab-slug"
LAB_TAG_VALUE = LAB_ID
REGION_PRIMARY = "us-east-1"
REGION_SECONDARY = "us-west-2"
LAB_REGIONS = [REGION_PRIMARY, REGION_SECONDARY]

# Resource names. Each regional resource carries a -use1 or -usw2 suffix.
EC2_PRIMARY_NAME = f"labrun-{LAB_ID}-ec2-use1"
EC2_SECONDARY_NAME = f"labrun-{LAB_ID}-ec2-usw2"
TG_PRIMARY_NAME = f"labrun-{LAB_ID}-tg-use1"
TG_SECONDARY_NAME = f"labrun-{LAB_ID}-tg-usw2"
ALB_PRIMARY_NAME = f"labrun-{LAB_ID}-alb-use1"
ALB_SECONDARY_NAME = f"labrun-{LAB_ID}-alb-usw2"
SG_PRIMARY_NAME = f"labrun-{LAB_ID}-sg-use1"
SG_SECONDARY_NAME = f"labrun-{LAB_ID}-sg-usw2"
RECORD_NAME = "app.labrun-lab7.example."

LAB_STATE = {
    "session_start": None,
    "primary": {
        "region": REGION_PRIMARY,
        "vpc_id": None,
        "subnets": [],
        "ami_id": None,
        "sg_id": None,
        "ec2_id": None,
        "tg_arn": None,
        "alb_arn": None,
        "alb_dns": None,
        "listener_arn": None,
    },
    "secondary": {
        "region": REGION_SECONDARY,
        "vpc_id": None,
        "subnets": [],
        "ami_id": None,
        "sg_id": None,
        "ec2_id": None,
        "tg_arn": None,
        "alb_arn": None,
        "alb_dns": None,
        "listener_arn": None,
    },
    "hosted_zone_id": None,
    "hc_primary_id": None,
    "hc_secondary_id": None,
    "record_name": RECORD_NAME,
    "failover_observed_seconds": None,
}


def _client(service, region):
    return boto3.client(
        service,
        aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
        aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
        aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
        region_name=region,
    )

In [ ]:
# NBVAL_SKIP
# Register the session and validate AWS credentials.

session_token = getpass.getpass("Paste your session token from labrun.dev: ")
aws_access_key_id = getpass.getpass("AWS_ACCESS_KEY_ID: ")
aws_secret_access_key = getpass.getpass("AWS_SECRET_ACCESS_KEY: ")
aws_session_token = getpass.getpass(
    "AWS_SESSION_TOKEN (leave blank for long-lived credentials): "
).strip()
region_input = input(f"AWS region for control plane (default {REGION_PRIMARY}): ").strip()
if region_input and region_input != REGION_PRIMARY:
    print(f"Region {region_input!r} is not supported as the primary region for this lab.")
    raise SystemExit(1)

AWS_CREDENTIALS = {
    "aws_access_key_id": aws_access_key_id,
    "aws_secret_access_key": aws_secret_access_key,
    "region": REGION_PRIMARY,
}
if aws_session_token:
    AWS_CREDENTIALS["aws_session_token"] = aws_session_token

sts = boto3.client(
    "sts",
    aws_access_key_id=aws_access_key_id,
    aws_secret_access_key=aws_secret_access_key,
    aws_session_token=aws_session_token or None,
    region_name=REGION_PRIMARY,
)
try:
    identity = sts.get_caller_identity()
except ClientError as e:
    print("Credentials are missing or expired. STS GetCallerIdentity failed:")
    print(f"  {e}")
    raise SystemExit(1)

ACCOUNT_ID = identity["Account"]
LAB_STATE["record_name"] = f"app.labrun-{LAB_ID}-{ACCOUNT_ID[:6]}.example."
print(f"Credentials validated. Account: {ACCOUNT_ID}")
print(f"Primary region:   {REGION_PRIMARY}")
print(f"Secondary region: {REGION_SECONDARY}")
print(f"Lab record name:  {LAB_STATE['record_name']}")

# Resolve AL2023 ARM AMI in both regions; AMI IDs differ per region.
for side, region in (("primary", REGION_PRIMARY), ("secondary", REGION_SECONDARY)):
    ssm = boto3.client(
        "ssm",
        aws_access_key_id=aws_access_key_id,
        aws_secret_access_key=aws_secret_access_key,
        aws_session_token=aws_session_token or None,
        region_name=region,
    )
    ami_id = ssm.get_parameter(
        Name="/aws/service/ami-amazon-linux-latest/al2023-ami-kernel-default-arm64",
    )["Parameter"]["Value"]
    LAB_STATE[side]["ami_id"] = ami_id
    LAB_STATE[side]["region"] = region
    print(f"  {side:9}: {region}  AMI={ami_id}")

# Discover default VPC + two AZs in each region.
for side in ("primary", "secondary"):
    region = LAB_STATE[side]["region"]
    ec2 = boto3.client(
        "ec2",
        aws_access_key_id=aws_access_key_id,
        aws_secret_access_key=aws_secret_access_key,
        aws_session_token=aws_session_token or None,
        region_name=region,
    )
    default_vpcs = ec2.describe_vpcs(Filters=[{"Name": "isDefault", "Values": ["true"]}])["Vpcs"]
    if not default_vpcs:
        print(f"No default VPC in {region}. Lab 7 expects a default VPC in both regions.")
        raise SystemExit(1)
    LAB_STATE[side]["vpc_id"] = default_vpcs[0]["VpcId"]
    subnets = ec2.describe_subnets(
        Filters=[{"Name": "vpc-id", "Values": [LAB_STATE[side]["vpc_id"]]}],
    )["Subnets"]
    azs_seen = {}
    for sn in subnets:
        azs_seen.setdefault(sn["AvailabilityZone"], sn["SubnetId"])
    LAB_STATE[side]["subnets"] = [azs_seen[az] for az in sorted(azs_seen.keys())[:2]]
    print(f"  {side:9}: VPC={LAB_STATE[side]['vpc_id']}  subnets={LAB_STATE[side]['subnets']}")

LAB_STATE["session_start"] = dt.datetime.now(dt.timezone.utc)
register(session_token=session_token, lab_id=LAB_ID, credentials=AWS_CREDENTIALS)
print(f"Session registered for lab_id: {LAB_ID}")

In [ ]:
# NBVAL_SKIP
# Cleanup manifest, atexit safety net, multi-region orphan scan.
# Records and health checks must delete before the hosted zone; ALBs and
# instances must delete before security groups. Per Gap 1 in the research
# doc, every cross-region resource declares its region on CleanupResource.

CLEANUP_MANIFEST = []


def _rebuild_manifest():
    CLEANUP_MANIFEST.clear()
    if LAB_STATE["hosted_zone_id"]:
        zone_id = LAB_STATE["hosted_zone_id"]
        for side, failover, hc_key in (
            ("primary", "PRIMARY", "hc_primary_id"),
            ("secondary", "SECONDARY", "hc_secondary_id"),
        ):
            alb_arn = LAB_STATE[side]["alb_arn"]
            alb_dns = LAB_STATE[side]["alb_dns"]
            hc_id = LAB_STATE[hc_key]
            if hc_id and alb_arn and alb_dns:
                # Best-effort record reconstruction for the cleanup delete.
                alb_zone_id = _alb_canonical_zone_id(side)
                record_set = {
                    "Name": LAB_STATE["record_name"],
                    "Type": "A",
                    "SetIdentifier": f"labrun-{failover.lower()}",
                    "Failover": failover,
                    "AliasTarget": {
                        "HostedZoneId": alb_zone_id,
                        "DNSName": alb_dns + ".",
                        "EvaluateTargetHealth": True,
                    },
                    "HealthCheckId": hc_id,
                }
                CLEANUP_MANIFEST.append(
                    CleanupResource(
                        type="route53_record",
                        id=f"{LAB_STATE['record_name']}/{failover}",
                        region=REGION_PRIMARY,
                        extra={"HostedZoneId": zone_id, "ResourceRecordSet": record_set},
                        cli_delete_command=(
                            f"aws route53 change-resource-record-sets "
                            f"--hosted-zone-id {zone_id} "
                            f"--change-batch '<delete batch for {failover} record>'"
                        ),
                    )
                )
        if LAB_STATE["hc_primary_id"]:
            CLEANUP_MANIFEST.append(
                CleanupResource(
                    type="route53_health_check",
                    id=LAB_STATE["hc_primary_id"],
                    region=REGION_PRIMARY,
                    cli_delete_command=(
                        f"aws route53 delete-health-check "
                        f"--health-check-id {LAB_STATE['hc_primary_id']}"
                    ),
                )
            )
        if LAB_STATE["hc_secondary_id"]:
            CLEANUP_MANIFEST.append(
                CleanupResource(
                    type="route53_health_check",
                    id=LAB_STATE["hc_secondary_id"],
                    region=REGION_PRIMARY,
                    cli_delete_command=(
                        f"aws route53 delete-health-check "
                        f"--health-check-id {LAB_STATE['hc_secondary_id']}"
                    ),
                )
            )
        CLEANUP_MANIFEST.append(
            CleanupResource(
                type="route53_hosted_zone",
                id=LAB_STATE["hosted_zone_id"],
                region=REGION_PRIMARY,
                cli_delete_command=(
                    f"aws route53 delete-hosted-zone --id {LAB_STATE['hosted_zone_id']}"
                ),
            )
        )

    for side in ("primary", "secondary"):
        region = LAB_STATE[side]["region"]
        s = LAB_STATE[side]
        if s["listener_arn"]:
            CLEANUP_MANIFEST.append(
                CleanupResource(
                    type="elbv2_listener",
                    id=s["listener_arn"],
                    region=region,
                    cli_delete_command=(
                        f"aws elbv2 delete-listener --listener-arn {s['listener_arn']} "
                        f"--region {region}"
                    ),
                )
            )
        if s["alb_arn"]:
            CLEANUP_MANIFEST.append(
                CleanupResource(
                    type="elbv2_load_balancer",
                    id=s["alb_arn"],
                    region=region,
                    cli_delete_command=(
                        f"aws elbv2 delete-load-balancer --load-balancer-arn {s['alb_arn']} "
                        f"--region {region}"
                    ),
                )
            )
        if s["tg_arn"]:
            CLEANUP_MANIFEST.append(
                CleanupResource(
                    type="elbv2_target_group",
                    id=s["tg_arn"],
                    region=region,
                    cli_delete_command=(
                        f"aws elbv2 delete-target-group --target-group-arn {s['tg_arn']} "
                        f"--region {region}"
                    ),
                )
            )
        if s["ec2_id"]:
            CLEANUP_MANIFEST.append(
                CleanupResource(
                    type="ec2_instance",
                    id=s["ec2_id"],
                    region=region,
                    cli_delete_command=(
                        f"aws ec2 terminate-instances --instance-ids {s['ec2_id']} --region {region}"
                    ),
                )
            )
        if s["sg_id"]:
            CLEANUP_MANIFEST.append(
                CleanupResource(
                    type="ec2_security_group",
                    id=s["sg_id"],
                    region=region,
                    cli_delete_command=(
                        f"aws ec2 delete-security-group --group-id {s['sg_id']} --region {region}"
                    ),
                )
            )


def _alb_canonical_zone_id(side):
    """AWS-published canonical hosted zone IDs for ALB alias targets."""
    region = LAB_STATE[side]["region"]
    # Documented in https://docs.aws.amazon.com/general/latest/gr/elb.html
    zones = {
        "us-east-1": "Z35SXDOTRQ7X7K",
        "us-west-2": "Z1H1FL5HABSF5",
    }
    return zones.get(region, "Z35SXDOTRQ7X7K")


_rebuild_manifest()


def _atexit_cleanup():
    try:
        _rebuild_manifest()
        run_cleanup(CLEANUP_MANIFEST)
    except Exception:
        pass


atexit.register(_atexit_cleanup)


def scan_for_orphans():
    arns = []
    for region in LAB_REGIONS:
        client = _client("resourcegroupstaggingapi", region)
        paginator = client.get_paginator("get_resources")
        for page in paginator.paginate(
            TagFilters=[{"Key": LAB_TAG_KEY, "Values": [LAB_TAG_VALUE]}],
        ):
            for item in page.get("ResourceTagMappingList", []):
                arns.append(item.get("ResourceARN", ""))
    return arns


orphans = scan_for_orphans()
if orphans:
    print(f"BLOCKED: existing resources tagged labrun:lab-slug={LAB_TAG_VALUE} found:")
    for arn in orphans:
        print("  -", arn)
    print()
    print("Run the cleanup cell at the bottom of this notebook first, then re-run setup.")
    raise SystemExit(1)

print("No prior orphans found in either region. Safe to provision.")

## Task 1: Stand up matching ALB + EC2 stacks in each region

A helper builds the same shape in both regions: SG, target group, ALB, listener, EC2 instance running an HTTP server whose response body names the region. The helper takes a region argument and runs the same flow against the regional boto3 clients.

For each region:

1. Create a security group in the default VPC with inbound 80 from anywhere.
2. Create a target group (HTTP/80, health check on `/`, `TargetType=instance`).
3. Create the ALB across two AZ-distinct subnets.
4. Create a listener forwarding to the target group.
5. Launch a t4g.nano EC2 instance whose user-data writes the region name into the served HTML and starts a Python HTTP server.
6. Register the instance with the target group via `register_targets`.

The observe cell polls both regions in parallel until both targets reach `healthy`. Ceiling 8 minutes (instance launch + status checks + ALB health check in each region, in parallel).

In [ ]:
# NBVAL_SKIP
# Task 1: build the regional stack helper and run it for both regions.

def _user_data_for(region):
    body = (
        f"<html><body>"
        f"labrun route53 failover lab. Region: {region}. "
        f"Hostname: $(hostname)."
        f"</body></html>"
    )
    script = (
        "#!/bin/bash\n"
        "set -e\n"
        f"mkdir -p /var/lib/labrun\n"
        f"cat > /var/lib/labrun/index.html <<'BODY'\n"
        f"{body}\n"
        f"BODY\n"
        f"cd /var/lib/labrun\n"
        f"nohup python3 -m http.server 80 > /var/log/labrun.log 2>&1 &\n"
    )
    return base64.b64encode(script.encode("utf-8")).decode("utf-8")


def build_regional_stack(side):
    """Provision the regional SG/TG/ALB/Listener/EC2 stack for the given side."""
    s = LAB_STATE[side]
    region = s["region"]
    ec2c = _client("ec2", region)
    elbv2c = _client("elbv2", region)

    # Security group (inbound 80, outbound all).
    sg_name = SG_PRIMARY_NAME if side == "primary" else SG_SECONDARY_NAME
    try:
        sg = ec2c.create_security_group(
            GroupName=sg_name,
            Description=f"labrun lab 7 SG {region}",
            VpcId=s["vpc_id"],
            TagSpecifications=[
                {
                    "ResourceType": "security-group",
                    "Tags": [{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
                }
            ],
        )
        s["sg_id"] = sg["GroupId"]
        ec2c.authorize_security_group_ingress(
            GroupId=s["sg_id"],
            IpPermissions=[
                {
                    "IpProtocol": "tcp",
                    "FromPort": 80,
                    "ToPort": 80,
                    "IpRanges": [{"CidrIp": "0.0.0.0/0"}],
                }
            ],
        )
    except ClientError as e:
        if e.response.get("Error", {}).get("Code") == "InvalidGroup.Duplicate":
            sgs = ec2c.describe_security_groups(
                Filters=[{"Name": "group-name", "Values": [sg_name]}],
            )["SecurityGroups"]
            s["sg_id"] = sgs[0]["GroupId"]
        else:
            raise

    # Target group.
    tg_name = TG_PRIMARY_NAME if side == "primary" else TG_SECONDARY_NAME
    # YOUR CODE: call elbv2c.create_target_group with Name=tg_name,
    # Protocol="HTTP", Port=80, VpcId=s["vpc_id"], TargetType="instance",
    # HealthCheckProtocol="HTTP", HealthCheckPath="/",
    # HealthCheckIntervalSeconds=15, HealthyThresholdCount=2,
    # Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}]. Capture
    # TargetGroupArn in s["tg_arn"]. Handle DuplicateTargetGroupName by
    # re-describing by name.

    # ALB.
    alb_name = ALB_PRIMARY_NAME if side == "primary" else ALB_SECONDARY_NAME
    # YOUR CODE: call elbv2c.create_load_balancer with Name=alb_name,
    # Subnets=s["subnets"], SecurityGroups=[s["sg_id"]], Scheme="internet-facing",
    # Type="application", IpAddressType="ipv4", Tags=[{"Key": LAB_TAG_KEY,
    # "Value": LAB_TAG_VALUE}]. Capture LoadBalancerArn in s["alb_arn"] and
    # DNSName in s["alb_dns"]. Handle DuplicateLoadBalancerName as a no-op.

    # Listener.
    # YOUR CODE: call elbv2c.create_listener with LoadBalancerArn=s["alb_arn"],
    # Protocol="HTTP", Port=80, DefaultActions=[{"Type": "forward",
    # "TargetGroupArn": s["tg_arn"]}], Tags=[{"Key": LAB_TAG_KEY,
    # "Value": LAB_TAG_VALUE}]. Capture ListenerArn in s["listener_arn"].

    # EC2 instance.
    # YOUR CODE: call ec2c.run_instances with ImageId=s["ami_id"],
    # InstanceType="t4g.nano", MinCount=1, MaxCount=1, SubnetId=s["subnets"][0],
    # SecurityGroupIds=[s["sg_id"]], UserData=_user_data_for(region),
    # TagSpecifications=[{"ResourceType": "instance", "Tags": [
    #     {"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE},
    #     {"Key": "Name", "Value": EC2_PRIMARY_NAME if side == "primary" else EC2_SECONDARY_NAME},
    # ]}]. Capture InstanceId in s["ec2_id"].

    # Register the instance with the target group.
    if s["ec2_id"] and s["tg_arn"]:
        elbv2c.register_targets(
            TargetGroupArn=s["tg_arn"],
            Targets=[{"Id": s["ec2_id"], "Port": 80}],
        )

    return s


build_regional_stack("primary")
build_regional_stack("secondary")
_rebuild_manifest()

print("Regional stacks requested in both regions:")
for side in ("primary", "secondary"):
    s = LAB_STATE[side]
    print(f"  {side:9}: region={s['region']}  ALB={s['alb_arn']}  EC2={s['ec2_id']}")

In [ ]:
# NBVAL_SKIP
# Observe: poll target health in both regions until each region has at
# least one healthy target. Ceiling 8 minutes.

deadline = time.time() + 480
while time.time() < deadline:
    clear_output(wait=True)
    rows = []
    healthy_count = {"primary": False, "secondary": False}
    for side in ("primary", "secondary"):
        s = LAB_STATE[side]
        region = s["region"]
        elbv2c = _client("elbv2", region)
        try:
            health = elbv2c.describe_target_health(TargetGroupArn=s["tg_arn"])["TargetHealthDescriptions"]
        except ClientError as e:
            health = []
            rows.append((region, "error", str(e)))
            continue
        for h in health:
            t_id = (h.get("Target") or {}).get("Id", "?")
            t_state = (h.get("TargetHealth") or {}).get("State", "?")
            rows.append((region, t_id, t_state))
            if t_state == "healthy":
                healthy_count[side] = True
    elapsed = int(480 - (deadline - time.time()))
    print(f"Regional target health at t+{elapsed:>3}s:")
    print("-" * 64)
    print(f"  {'region':12}  {'instance':20}  {'state'}")
    print("-" * 64)
    for region, t_id, state in rows:
        print(f"  {region:12}  {t_id:20}  {state}")
    if all(healthy_count.values()):
        print()
        print("Both regional stacks are healthy. Move on to Task 2.")
        break
    time.sleep(20)
else:
    print()
    print("Timed out before both regions had healthy targets.")

In [ ]:
# NBVAL_SKIP
# Checkpoint 1: us-east-1 ALB exists, healthy, response body contains the region.

def checkpoint_1(session):
    from labrun_checks import CheckpointResult
    try:
        s = LAB_STATE["primary"]
        elbv2_c = _client("elbv2", REGION_PRIMARY)
        alb = elbv2_c.describe_load_balancers(LoadBalancerArns=[s["alb_arn"]])["LoadBalancers"][0]
        if alb.get("State", {}).get("Code") != "active":
            return CheckpointResult(
                status="fail",
                error_reason=f"us-east-1 ALB state is {alb.get('State', {}).get('Code')!r}.",
            )
        health = elbv2_c.describe_target_health(TargetGroupArn=s["tg_arn"])["TargetHealthDescriptions"]
        healthy = [h for h in health if (h.get("TargetHealth") or {}).get("State") == "healthy"]
        if not healthy:
            return CheckpointResult(
                status="fail",
                error_reason="No healthy target in us-east-1 target group.",
            )
        try:
            with urlopen(f"http://{s['alb_dns']}/", timeout=5) as resp:
                body = resp.read(2048).decode("utf-8", errors="replace")
        except Exception as e:
            return CheckpointResult(
                status="fail",
                error_reason=f"Could not HTTP GET us-east-1 ALB: {e}",
            )
        if "us-east-1" not in body:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    "Response from us-east-1 ALB does not contain the substring "
                    f"'us-east-1'. Got: {body[:200]!r}"
                ),
            )
        return CheckpointResult(status="pass")
    except (ClientError, IndexError, KeyError) as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(1, checkpoint_1)

<details><summary>Hint 1 (nudge)</summary>

The helper takes a `side` argument and calls regional clients via `_client(service, region)`. Every API call in the helper must use the regional client; do not reuse the us-east-1 client when `side="secondary"`.

</details>

<details><summary>Hint 2 (direction)</summary>

Four sequential calls per region: target group, ALB, listener, run_instances. Then one extra call (`register_targets`) wires the instance into the target group so the ALB starts health-checking it. The lab already gives you `_user_data_for(region)`, the security group, the AMI ID, and the subnets via `LAB_STATE[side]`.

</details>

<details><summary>Hint 3 (spoiler)</summary>

```python
tg = elbv2c.create_target_group(
    Name=tg_name,
    Protocol="HTTP",
    Port=80,
    VpcId=s["vpc_id"],
    TargetType="instance",
    HealthCheckProtocol="HTTP",
    HealthCheckPath="/",
    HealthCheckIntervalSeconds=15,
    HealthyThresholdCount=2,
    Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
)["TargetGroups"][0]
s["tg_arn"] = tg["TargetGroupArn"]

alb = elbv2c.create_load_balancer(
    Name=alb_name,
    Subnets=s["subnets"],
    SecurityGroups=[s["sg_id"]],
    Scheme="internet-facing",
    Type="application",
    IpAddressType="ipv4",
    Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
)["LoadBalancers"][0]
s["alb_arn"] = alb["LoadBalancerArn"]
s["alb_dns"] = alb["DNSName"]

listener = elbv2c.create_listener(
    LoadBalancerArn=s["alb_arn"],
    Protocol="HTTP",
    Port=80,
    DefaultActions=[{"Type": "forward", "TargetGroupArn": s["tg_arn"]}],
    Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
)["Listeners"][0]
s["listener_arn"] = listener["ListenerArn"]

resp = ec2c.run_instances(
    ImageId=s["ami_id"],
    InstanceType="t4g.nano",
    MinCount=1, MaxCount=1,
    SubnetId=s["subnets"][0],
    SecurityGroupIds=[s["sg_id"]],
    UserData=_user_data_for(region),
    TagSpecifications=[{
        "ResourceType": "instance",
        "Tags": [
            {"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE},
            {"Key": "Name", "Value": EC2_PRIMARY_NAME if side == "primary" else EC2_SECONDARY_NAME},
        ],
    }],
)
s["ec2_id"] = resp["Instances"][0]["InstanceId"]
```

</details>

**Wallet check.** Two ALBs just started billing at $0.0225/hour each. About 0.075 cents per minute combined. Two t4g.nano are sub-cent per hour. Running total at the end of Task 1 is roughly 3 cents.

## Task 2: Confirm the us-west-2 stack is healthy

Same checkpoint shape as Task 1 but for the secondary region. This is a code_observe task with no new student code; the observe cell from Task 1 already verifies both regions are healthy.

In [ ]:
# NBVAL_SKIP
# Task 2: no new student code. Re-run the observe block from Task 1 if
# either region is not yet healthy; the checkpoint below verifies the
# us-west-2 stack independently.

print("Re-running the regional health snapshot for us-west-2.")
s = LAB_STATE["secondary"]
elbv2_c = _client("elbv2", REGION_SECONDARY)
try:
    health = elbv2_c.describe_target_health(TargetGroupArn=s["tg_arn"])["TargetHealthDescriptions"]
except ClientError as e:
    health = []
    print(f"Error: {e}")
print(f"  us-west-2 targets:")
for h in health:
    t_id = (h.get("Target") or {}).get("Id", "?")
    t_state = (h.get("TargetHealth") or {}).get("State", "?")
    print(f"    {t_id:20}  state={t_state}")

In [ ]:
# NBVAL_SKIP
# Observe: snapshot the us-west-2 ALB DNS resolution and response body so
# the student sees Architecture B is genuinely standing up traffic.

s = LAB_STATE["secondary"]
clear_output(wait=True)
print(f"us-west-2 ALB:")
print(f"  ARN:  {s['alb_arn']}")
print(f"  DNS:  {s['alb_dns']}")
try:
    with urlopen(f"http://{s['alb_dns']}/", timeout=5) as resp:
        body = resp.read(512).decode("utf-8", errors="replace")
    print(f"  Body (first 200 chars):")
    print(f"    {body[:200]}")
except Exception as e:
    print(f"  HTTP GET failed: {e}")

In [ ]:
# NBVAL_SKIP
# Checkpoint 2: us-west-2 ALB exists, healthy, response body contains 'us-west-2'.

def checkpoint_2(session):
    from labrun_checks import CheckpointResult
    try:
        s = LAB_STATE["secondary"]
        elbv2_c = _client("elbv2", REGION_SECONDARY)
        alb = elbv2_c.describe_load_balancers(LoadBalancerArns=[s["alb_arn"]])["LoadBalancers"][0]
        if alb.get("State", {}).get("Code") != "active":
            return CheckpointResult(
                status="fail",
                error_reason=f"us-west-2 ALB state is {alb.get('State', {}).get('Code')!r}.",
            )
        health = elbv2_c.describe_target_health(TargetGroupArn=s["tg_arn"])["TargetHealthDescriptions"]
        healthy = [h for h in health if (h.get("TargetHealth") or {}).get("State") == "healthy"]
        if not healthy:
            return CheckpointResult(
                status="fail",
                error_reason="No healthy target in us-west-2 target group.",
            )
        try:
            with urlopen(f"http://{s['alb_dns']}/", timeout=5) as resp:
                body = resp.read(2048).decode("utf-8", errors="replace")
        except Exception as e:
            return CheckpointResult(
                status="fail",
                error_reason=f"Could not HTTP GET us-west-2 ALB: {e}",
            )
        if "us-west-2" not in body:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    "Response from us-west-2 ALB does not contain 'us-west-2'. "
                    f"Got: {body[:200]!r}"
                ),
            )
        return CheckpointResult(status="pass")
    except (ClientError, IndexError, KeyError) as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(2, checkpoint_2)

<details><summary>Hint 1 (nudge)</summary>

Nothing new to build. `build_regional_stack("secondary")` already ran in Task 1's cell; this checkpoint just verifies the result.

</details>

<details><summary>Hint 2 (direction)</summary>

If the checkpoint fails, the instance in us-west-2 probably has not finished user-data yet. Re-run the observe cell from Task 1 and watch the `us-west-2` row until state=healthy. The check then passes without code changes.

</details>

<details><summary>Hint 3 (spoiler)</summary>

No spoiler needed. Re-run the observe cell from Task 1 and re-run this checkpoint.

</details>

**Wallet check.** No new line items vs Task 1. Both ALBs still billing combined at ~$0.045/hour.

## Task 3: Create the hosted zone, two health checks, and the two failover records

Three groups of calls:

1. `route53.create_hosted_zone(Name="labrun-<account>.example.", VPC=None)` for a **public** hosted zone. The lab does not register a real domain; you use `test_dns_answer` to query the authoritative resolvers directly. Tag the zone.
2. `route53.create_health_check` twice, once per regional ALB. The endpoint type is `HTTP`, the resource path is `/`, the port is 80, and the fully-qualified domain name is the ALB's regional DNS.
3. `route53.change_resource_record_sets` with two `CREATE` actions: a PRIMARY failover A-alias record pointing at the us-east-1 ALB, and a SECONDARY failover A-alias record pointing at the us-west-2 ALB. Each record carries a `HealthCheckId`. Both records share the same `Name`; the `SetIdentifier` and `Failover` fields distinguish them.

ALB alias records need the canonical hosted zone ID per region (already provided by `_alb_canonical_zone_id`).

In [ ]:
# NBVAL_SKIP
# Task 3: hosted zone + health checks + failover records.

r53 = _client("route53", REGION_PRIMARY)
zone_name = LAB_STATE["record_name"].rsplit(".", maxsplit=1)[0]
zone_name = zone_name.split(".", maxsplit=1)[1] + "."
print(f"Hosted zone name: {zone_name}")

# Create the public hosted zone.
# YOUR CODE: call r53.create_hosted_zone(Name=zone_name,
# CallerReference=f"labrun-{LAB_ID}-{int(time.time())}",
# HostedZoneConfig={"Comment": "labrun lab 7 failover routing",
# "PrivateZone": False}). Capture the zone ID (strip the /hostedzone/ prefix)
# in LAB_STATE["hosted_zone_id"]. Tag the zone with route53.change_tags_for_resource
# (ResourceType="hostedzone").

# Create both health checks.
def create_hc(side):
    s = LAB_STATE[side]
    # YOUR CODE: call r53.create_health_check(
    #     CallerReference=f"labrun-{LAB_ID}-{side}-{int(time.time())}",
    #     HealthCheckConfig={
    #         "Type": "HTTP",
    #         "Port": 80,
    #         "ResourcePath": "/",
    #         "FullyQualifiedDomainName": s["alb_dns"],
    #         "RequestInterval": 30,
    #         "FailureThreshold": 3,
    #     },
    # ). Capture the HealthCheckId; tag it with the lab slug.
    return None  # YOUR CODE replaces with the returned HealthCheckId


LAB_STATE["hc_primary_id"] = create_hc("primary")
LAB_STATE["hc_secondary_id"] = create_hc("secondary")

# Create the two failover records as a single change batch.
def _alb_record(side, failover):
    s = LAB_STATE[side]
    hc_id = LAB_STATE["hc_primary_id"] if side == "primary" else LAB_STATE["hc_secondary_id"]
    return {
        "Action": "CREATE",
        "ResourceRecordSet": {
            "Name": LAB_STATE["record_name"],
            "Type": "A",
            "SetIdentifier": f"labrun-{failover.lower()}",
            "Failover": failover,
            "AliasTarget": {
                "HostedZoneId": _alb_canonical_zone_id(side),
                "DNSName": s["alb_dns"] + ".",
                "EvaluateTargetHealth": True,
            },
            "HealthCheckId": hc_id,
        },
    }


# YOUR CODE: call r53.change_resource_record_sets(
#     HostedZoneId=LAB_STATE["hosted_zone_id"],
#     ChangeBatch={"Changes": [
#         _alb_record("primary", "PRIMARY"),
#         _alb_record("secondary", "SECONDARY"),
#     ]},
# ). Print the returned ChangeInfo.Status.

_rebuild_manifest()
print(f"Hosted zone:        {LAB_STATE['hosted_zone_id']}")
print(f"Health check 1:     {LAB_STATE['hc_primary_id']} (us-east-1)")
print(f"Health check 2:     {LAB_STATE['hc_secondary_id']} (us-west-2)")
print(f"Records:            two failover A-aliases for {LAB_STATE['record_name']}")

In [ ]:
# NBVAL_SKIP
# Observe: poll the health checks until both publish their first status
# results (typically 30-90 seconds after creation). Ceiling 3 minutes.

r53_obs = _client("route53", REGION_PRIMARY)
deadline = time.time() + 180
while time.time() < deadline:
    clear_output(wait=True)
    rows = []
    for side, label in (("primary", "us-east-1"), ("secondary", "us-west-2")):
        hc_id = LAB_STATE["hc_primary_id"] if side == "primary" else LAB_STATE["hc_secondary_id"]
        if not hc_id:
            rows.append((label, hc_id, "no health check id"))
            continue
        try:
            status = r53_obs.get_health_check_status(HealthCheckId=hc_id)
            observations = status.get("HealthCheckObservations", [])
            if observations:
                states = [
                    (o.get("StatusReport") or {}).get("Status", "?")
                    for o in observations
                ]
                rows.append((label, hc_id, ", ".join(states[:3])))
            else:
                rows.append((label, hc_id, "initializing"))
        except ClientError as e:
            rows.append((label, hc_id, f"error: {e}"))
    elapsed = int(180 - (deadline - time.time()))
    print(f"Health check status at t+{elapsed:>3}s:")
    print("-" * 78)
    for region_label, hc_id, status in rows:
        print(f"  {region_label:12}  {hc_id:36}  {status}")
    settled = all("Success" in r[2] or "Failure" in r[2] for r in rows)
    if settled:
        print()
        print("Both health checks have published initial status. Move on.")
        break
    time.sleep(15)
else:
    print()
    print("Some health checks did not publish initial status. Re-run Checkpoint 3.")

In [ ]:
# NBVAL_SKIP
# Checkpoint 3: hosted zone exists, two failover records present, two health
# checks attached.

def checkpoint_3(session):
    from labrun_checks import CheckpointResult
    try:
        r53_c = _client("route53", REGION_PRIMARY)
        zone = r53_c.get_hosted_zone(Id=LAB_STATE["hosted_zone_id"])["HostedZone"]
        if not zone.get("Config", {}).get("PrivateZone", True) is False:
            return CheckpointResult(
                status="fail",
                error_reason="Hosted zone is private; lab uses a public zone for test_dns_answer.",
            )

        records = r53_c.list_resource_record_sets(
            HostedZoneId=LAB_STATE["hosted_zone_id"],
            StartRecordName=LAB_STATE["record_name"],
            StartRecordType="A",
            MaxItems="20",
        )["ResourceRecordSets"]
        primary_rec = next(
            (r for r in records if r.get("Name") == LAB_STATE["record_name"]
             and r.get("Failover") == "PRIMARY"),
            None,
        )
        secondary_rec = next(
            (r for r in records if r.get("Name") == LAB_STATE["record_name"]
             and r.get("Failover") == "SECONDARY"),
            None,
        )
        if primary_rec is None or secondary_rec is None:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    "Expected one PRIMARY and one SECONDARY failover record on "
                    f"{LAB_STATE['record_name']!r}. Found: "
                    f"{[r.get('Failover') for r in records]}"
                ),
            )
        for label, rec in (("PRIMARY", primary_rec), ("SECONDARY", secondary_rec)):
            if not rec.get("HealthCheckId"):
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"{label} record is missing HealthCheckId. Without it, "
                        f"Route 53 treats the record as always-healthy and never flips."
                    ),
                )
        return CheckpointResult(status="pass")
    except ClientError as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(3, checkpoint_3)

<details><summary>Hint 1 (nudge)</summary>

Three Route 53 calls. `create_hosted_zone` returns an Id with a `/hostedzone/` prefix; strip it. `create_health_check` returns a `HealthCheck.Id`. `change_resource_record_sets` takes a list of `Changes` with `Action` and `ResourceRecordSet`.

</details>

<details><summary>Hint 2 (direction)</summary>

For the records, the AWS-recommended pattern for ALB targets is an A-alias record (not an A record with an IP). The `AliasTarget` needs three fields: `HostedZoneId` (the canonical zone ID for ALB in that region, already provided), `DNSName` (the regional ALB's DNS name with a trailing dot), and `EvaluateTargetHealth=True`. Both records share the same `Name` so Route 53 picks one or the other based on `Failover` and health check state.

</details>

<details><summary>Hint 3 (spoiler)</summary>

```python
resp = r53.create_hosted_zone(
    Name=zone_name,
    CallerReference=f"labrun-{LAB_ID}-{int(time.time())}",
    HostedZoneConfig={"Comment": "labrun lab 7 failover routing", "PrivateZone": False},
)
LAB_STATE["hosted_zone_id"] = resp["HostedZone"]["Id"].split("/")[-1]
r53.change_tags_for_resource(
    ResourceType="hostedzone",
    ResourceId=LAB_STATE["hosted_zone_id"],
    AddTags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
)

def create_hc(side):
    s = LAB_STATE[side]
    resp = r53.create_health_check(
        CallerReference=f"labrun-{LAB_ID}-{side}-{int(time.time())}",
        HealthCheckConfig={
            "Type": "HTTP",
            "Port": 80,
            "ResourcePath": "/",
            "FullyQualifiedDomainName": s["alb_dns"],
            "RequestInterval": 30,
            "FailureThreshold": 3,
        },
    )
    hc_id = resp["HealthCheck"]["Id"]
    r53.change_tags_for_resource(
        ResourceType="healthcheck",
        ResourceId=hc_id,
        AddTags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
    )
    return hc_id

LAB_STATE["hc_primary_id"] = create_hc("primary")
LAB_STATE["hc_secondary_id"] = create_hc("secondary")

resp = r53.change_resource_record_sets(
    HostedZoneId=LAB_STATE["hosted_zone_id"],
    ChangeBatch={"Changes": [
        _alb_record("primary", "PRIMARY"),
        _alb_record("secondary", "SECONDARY"),
    ]},
)
print("ChangeInfo:", resp["ChangeInfo"]["Status"])
```

</details>

**Wallet check.** Route 53 hosted zone: $0.50/month, prorated to fractions of a cent for the lab session. Two health checks at $0.50/check/month after the first 50 free. Session burn from Task 3 is negligible compared to the ALB hours.

## Task 4: Verify test_dns_answer returns the us-east-1 ALB while both are healthy

`route53.test_dns_answer(HostedZoneId=..., RecordName=..., RecordType="A")` asks the Route 53 authoritative resolvers what they would return for a given query. It does not consult a public resolver cache, so the answer is immediate.

While both health checks publish `Success`, Route 53 returns the PRIMARY failover record (the us-east-1 ALB's IPs). The validator captures the ALB's resolved IPs via Python's `socket.gethostbyname_ex` and asserts the `test_dns_answer` result intersects them.

Health-check propagation lag: a newly created health check takes 30-90 seconds to publish its first result. The observe cell polls `test_dns_answer` every 15 seconds for up to 90 seconds.

In [ ]:
# NBVAL_SKIP
# Task 4: query test_dns_answer until it returns the us-east-1 ALB's IPs.

r53 = _client("route53", REGION_PRIMARY)


def _alb_ips(dns_name):
    """Return the set of IPs the public DNS currently resolves the ALB to."""
    try:
        _, _, ips = socket.gethostbyname_ex(dns_name)
        return set(ips)
    except Exception:
        return set()


# YOUR CODE: call r53.test_dns_answer(HostedZoneId=LAB_STATE["hosted_zone_id"],
# RecordName=LAB_STATE["record_name"], RecordType="A"). Read the
# RecordData list (an array of IP strings or DNS names depending on the
# alias target). For an ALB alias, RecordData often contains nothing useful;
# instead call socket.gethostbyname_ex on the ALB DNS name to get IPs.
# Print both the test_dns_answer result and the resolved IPs.

resp = r53.test_dns_answer(
    HostedZoneId=LAB_STATE["hosted_zone_id"],
    RecordName=LAB_STATE["record_name"],
    RecordType="A",
)
print(f"test_dns_answer RecordData: {resp.get('RecordData')}")
print(f"Nameserver:                 {resp.get('Nameserver')}")
print()
primary_ips = _alb_ips(LAB_STATE["primary"]["alb_dns"])
secondary_ips = _alb_ips(LAB_STATE["secondary"]["alb_dns"])
print(f"us-east-1 ALB resolves to:  {sorted(primary_ips)}")
print(f"us-west-2 ALB resolves to:  {sorted(secondary_ips)}")

In [ ]:
# NBVAL_SKIP
# Observe: poll test_dns_answer every 15 seconds until the RecordData
# matches the us-east-1 ALB's resolved IPs. Ceiling 90 seconds.

r53_obs = _client("route53", REGION_PRIMARY)
deadline = time.time() + 90
target_side = "primary"
matched = False
while time.time() < deadline:
    clear_output(wait=True)
    primary_ips = _alb_ips(LAB_STATE["primary"]["alb_dns"])
    secondary_ips = _alb_ips(LAB_STATE["secondary"]["alb_dns"])
    try:
        resp = r53_obs.test_dns_answer(
            HostedZoneId=LAB_STATE["hosted_zone_id"],
            RecordName=LAB_STATE["record_name"],
            RecordType="A",
        )
        record_data = set(resp.get("RecordData", []))
    except ClientError as e:
        record_data = set()
        print(f"test_dns_answer error: {e}")
    elapsed = int(90 - (deadline - time.time()))
    print(f"test_dns_answer poll at t+{elapsed:>3}s:")
    print("-" * 64)
    print(f"  Returned IPs:        {sorted(record_data)}")
    print(f"  us-east-1 ALB IPs:   {sorted(primary_ips)}")
    print(f"  us-west-2 ALB IPs:   {sorted(secondary_ips)}")
    if record_data and primary_ips and (record_data & primary_ips):
        print()
        print("Route 53 is returning the us-east-1 ALB (primary path healthy).")
        matched = True
        break
    time.sleep(15)
else:
    if not matched:
        print()
        print("Timed out before Route 53 returned the us-east-1 ALB.")

In [ ]:
# NBVAL_SKIP
# Checkpoint 4: test_dns_answer returns the us-east-1 ALB while both health
# checks are healthy.

def checkpoint_4(session):
    from labrun_checks import CheckpointResult
    try:
        r53_c = _client("route53", REGION_PRIMARY)
        primary_ips = _alb_ips(LAB_STATE["primary"]["alb_dns"])
        if not primary_ips:
            return CheckpointResult(
                status="fail",
                error_reason="us-east-1 ALB DNS did not resolve to any IPs.",
            )
        # Retry test_dns_answer for up to 60 seconds because health-check
        # propagation can lag.
        deadline = time.time() + 60
        last_returned = set()
        while time.time() < deadline:
            try:
                resp = r53_c.test_dns_answer(
                    HostedZoneId=LAB_STATE["hosted_zone_id"],
                    RecordName=LAB_STATE["record_name"],
                    RecordType="A",
                )
                returned = set(resp.get("RecordData", []))
            except ClientError as e:
                return CheckpointResult(status="error", error_reason=str(e))
            if returned & primary_ips:
                return CheckpointResult(status="pass")
            last_returned = returned
            time.sleep(10)
        return CheckpointResult(
            status="fail",
            error_reason=(
                f"test_dns_answer returned {sorted(last_returned)} which does not "
                f"intersect us-east-1 ALB IPs {sorted(primary_ips)}. Wait 60 more "
                f"seconds for health-check propagation and re-run."
            ),
        )
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(4, checkpoint_4)

<details><summary>Hint 1 (nudge)</summary>

One call: `route53.test_dns_answer`. It returns `RecordData`, a list of strings. For A-alias records, the strings are usually the resolved IPs of the alias target.

</details>

<details><summary>Hint 2 (direction)</summary>

If `test_dns_answer` returns the us-west-2 ALB's IPs instead of us-east-1, the primary health check probably has not finished publishing its first `Success` result yet. Wait 30-60 seconds and run the observe cell again.

</details>

<details><summary>Hint 3 (spoiler)</summary>

```python
resp = r53.test_dns_answer(
    HostedZoneId=LAB_STATE["hosted_zone_id"],
    RecordName=LAB_STATE["record_name"],
    RecordType="A",
)
print(f"test_dns_answer RecordData: {resp.get('RecordData')}")
```

</details>

**Wallet check.** `test_dns_answer` is free. Running total roughly 5-6 cents from ALB hours.

## Task 5: Break the primary health check and prove DNS flips to us-west-2

You change the us-east-1 target group's `HealthCheckPath` from `/` to a non-existent path. The ALB starts failing health checks within ~45 seconds (`HealthCheckIntervalSeconds=15` x `UnhealthyThresholdCount=3`). The Route 53 health check probing the ALB DNS sees 5xx responses (or no response) and transitions to `Failure` over the next 30-90 seconds.

Once the health check is `Failure`, Route 53 stops returning the PRIMARY record and starts returning the SECONDARY (us-west-2). The observe cell polls `test_dns_answer` every 15 seconds for up to 4 minutes and captures the wall-clock duration from "I broke it" to "DNS flipped.

In [ ]:
# NBVAL_SKIP
# Task 5: break the primary health check.

elbv2_use1 = _client("elbv2", REGION_PRIMARY)

# YOUR CODE: call elbv2_use1.modify_target_group(
#     TargetGroupArn=LAB_STATE["primary"]["tg_arn"],
#     HealthCheckPath="/labrun-fail-on-purpose",
# ). This makes the primary ALB fail its health check for every target;
# Route 53's health check then sees the ALB returning 5xx and transitions
# to Failure within 60-90 seconds.

LAB_STATE["primary_break_at"] = time.time()
print("Primary target group health-check path broken.")
print("Now run the observe cell to watch DNS flip to us-west-2.")

In [ ]:
# NBVAL_SKIP
# Observe: poll test_dns_answer every 15 seconds until the returned IPs
# match the us-west-2 ALB. Ceiling 4 minutes.

r53_obs = _client("route53", REGION_PRIMARY)
break_at = LAB_STATE.get("primary_break_at") or time.time()
deadline = time.time() + 240
flipped = False
while time.time() < deadline:
    clear_output(wait=True)
    primary_ips = _alb_ips(LAB_STATE["primary"]["alb_dns"])
    secondary_ips = _alb_ips(LAB_STATE["secondary"]["alb_dns"])
    try:
        resp = r53_obs.test_dns_answer(
            HostedZoneId=LAB_STATE["hosted_zone_id"],
            RecordName=LAB_STATE["record_name"],
            RecordType="A",
        )
        returned = set(resp.get("RecordData", []))
    except ClientError as e:
        returned = set()
    elapsed = int(time.time() - break_at)
    print(f"Failover progress at t+{elapsed:>3}s since break:")
    print("-" * 64)
    print(f"  test_dns_answer returns: {sorted(returned)}")
    print(f"  us-east-1 ALB:           {sorted(primary_ips)}")
    print(f"  us-west-2 ALB:           {sorted(secondary_ips)}")
    if returned and secondary_ips and (returned & secondary_ips) and not (returned & primary_ips):
        flipped = True
        LAB_STATE["failover_observed_seconds"] = elapsed
        print()
        print(f"DNS flipped to us-west-2 in {elapsed} seconds.")
        break
    time.sleep(15)
else:
    if not flipped:
        print()
        print("Timed out before DNS flipped. Health-check propagation can take")
        print("up to 90 seconds + ALB unhealthy threshold (~45s); 4 minutes is")
        print("usually enough but slow paths exist. Re-run if the AZ window did")
        print("not close in time.")

In [ ]:
# NBVAL_SKIP
# Checkpoint 5: test_dns_answer flipped from us-east-1 to us-west-2 within 3 minutes.

def checkpoint_5(session):
    from labrun_checks import CheckpointResult
    try:
        r53_c = _client("route53", REGION_PRIMARY)
        secondary_ips = _alb_ips(LAB_STATE["secondary"]["alb_dns"])
        primary_ips = _alb_ips(LAB_STATE["primary"]["alb_dns"])
        if not secondary_ips:
            return CheckpointResult(
                status="fail",
                error_reason="us-west-2 ALB DNS did not resolve to any IPs.",
            )
        # Retry test_dns_answer for up to 3 minutes from now.
        deadline = time.time() + 180
        last_returned = set()
        while time.time() < deadline:
            try:
                resp = r53_c.test_dns_answer(
                    HostedZoneId=LAB_STATE["hosted_zone_id"],
                    RecordName=LAB_STATE["record_name"],
                    RecordType="A",
                )
                returned = set(resp.get("RecordData", []))
            except ClientError as e:
                return CheckpointResult(status="error", error_reason=str(e))
            if returned & secondary_ips and not (returned & primary_ips):
                return CheckpointResult(status="pass")
            last_returned = returned
            time.sleep(10)
        return CheckpointResult(
            status="fail",
            error_reason=(
                f"test_dns_answer is still returning {sorted(last_returned)} "
                f"after the polling window. Expected intersection with "
                f"us-west-2 ALB IPs {sorted(secondary_ips)}."
            ),
        )
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(5, checkpoint_5)

<details><summary>Hint 1 (nudge)</summary>

One `elbv2.modify_target_group` call. The point of attack is the `HealthCheckPath`. The instance's HTTP server only serves `/`; pointing the target group at a path it does not serve makes every health check return 404.

</details>

<details><summary>Hint 2 (direction)</summary>

`elbv2.modify_target_group(TargetGroupArn=LAB_STATE["primary"]["tg_arn"], HealthCheckPath="/labrun-fail-on-purpose")`. After this lands, expect 45 seconds for the ALB-level target health to flip, then another 30-60 seconds for the Route 53 health check to register the failure, then up to 30 seconds for the DNS answer to flip.

</details>

<details><summary>Hint 3 (spoiler)</summary>

```python
elbv2_use1.modify_target_group(
    TargetGroupArn=LAB_STATE["primary"]["tg_arn"],
    HealthCheckPath="/labrun-fail-on-purpose",
)
```

</details>

**Wallet check.** Same per-hour burn through the failover window. The Route 53 health-check evaluation is included in the per-check monthly fee; no extra cost for the broken state.

## Cleanup

Cleanup deletes records first, then health checks, then the hosted zone (an empty zone is deletable), then both regional ALB/EC2 stacks. The cleanup adapter routes each delete to the right regional client via the `region` field on each CleanupResource. The multi-region orphan scan iterates both regions to catch anything tagged that escaped the manifest.

This is the highest-cleanup-risk lab in the SAA track. The hosted zone is the silent residual: $0.50/month forever if it is missed. The cleanup output explicitly names the zone in the success summary.

In [ ]:
# NBVAL_SKIP
# Cleanup. Multi-region, multi-service.

import sys

_rebuild_manifest()
result = run_cleanup(CLEANUP_MANIFEST)

for warning in result.warnings:
    print(warning)
for orphan in result.orphans:
    print(orphan)
if result.warnings or result.orphans:
    print()

failed_ids = set()
for w in result.warnings:
    for r in CLEANUP_MANIFEST:
        if r.id in w:
            failed_ids.add(r.id)
            break

critical_types = {"ec2_instance", "elbv2_load_balancer", "route53_hosted_zone"}
critical_total = sum(1 for r in CLEANUP_MANIFEST if r.type in critical_types)
critical_destroyed = sum(
    1 for r in CLEANUP_MANIFEST if r.type in critical_types and r.id not in failed_ids
)
standard_total = len(CLEANUP_MANIFEST) - critical_total
standard_destroyed = standard_total - sum(
    1 for rid in failed_ids for r in CLEANUP_MANIFEST
    if r.id == rid and r.type not in critical_types
)
failed_count = len(failed_ids)

print("Cleanup complete.")
print(f"Critical resources destroyed: {critical_destroyed}")
print(f"Standard resources destroyed: {standard_destroyed}")
print(f"Resources that failed to delete: {failed_count} (see above for CLI commands)")
print()
if LAB_STATE.get("hosted_zone_id"):
    print(f"Route 53 hosted zone {LAB_STATE['hosted_zone_id']} was in the manifest.")
    print(f"Confirm it is gone before closing the notebook.")
print("If K > 0, your AWS account may still incur charges. Resolve before closing this notebook.")

cleanup(status=result.status)

if failed_count > 0:
    sys.exit(1)

**Session total: approximately $0.10 to $0.30.** Two ALBs are the dominant line item, two t4g.nano are sub-cent each, the hosted zone is a fraction of a cent prorated. The cleanup cell took down both regions simultaneously; the multi-region orphan scan confirmed nothing escaped.

## Reflection

These are not graded. They are for you.

1. Walk through what Route 53 does in the 3 minutes after a primary endpoint's health check transitions to unhealthy. List the steps: health-check evaluator decides the endpoint is failed, propagation to the Route 53 authoritative resolvers, change to the answer returned by DNS queries, client DNS cache TTL expiration, client reconnect to the secondary. Where does the record TTL fit in that flow, and what TTL would you pick for a 60-second RTO versus a 5-minute RTO?

2. Your team is choosing between Route 53 failover routing (primary plus secondary with health checks), latency routing (Route 53 returns the lowest-latency endpoint), weighted routing (return endpoints in a ratio for blue-green or A/B tests), and geolocation routing (return endpoints based on the resolver's location). For three scenarios (a) DR with a cold standby region, (b) global users wanting lowest latency, (c) gradual 10% traffic shift to a new region: which routing policy fits each?

## Exam-Style Practice

**Question 1 (Domain 2, routing-policy selection by constraint):**

A SaaS company runs a multi-region web app with a primary in us-east-1 and a warm standby in eu-west-1. The SLA requires automatic traffic redirection within 5 minutes if the primary region's ALB stops responding to health checks. The team is choosing between four Route 53 routing policies. Which fits?

A. Simple routing with multiple values, returning both ALBs and letting the client retry.

B. Weighted routing with us-east-1 weighted 100 and eu-west-1 weighted 0, manually flipped when an operator detects the outage.

C. Failover routing with us-east-1 as PRIMARY and eu-west-1 as SECONDARY, each with a health check.

D. Geolocation routing with us-east-1 for North American users and eu-west-1 for European users.

<details><summary>Show answer</summary>

**Correct: C.**

- A is wrong: simple routing with multiple values returns all configured IPs and relies on the client to fail over. AWS does NOT perform health checks on simple-routing records by default. SLA unmet.
- B is wrong: weighted routing requires manual flipping, which is not automatic redirection. The "5 minutes if primary stops responding" requirement is automatic, not operator-triggered.
- C is correct: failover routing with PRIMARY plus SECONDARY records and health checks is the AWS-recommended pattern for active/standby DR. Route 53 evaluates the primary's health check, flips DNS to the secondary on failure, and propagation lands inside the 5-minute SLA.
- D is wrong: geolocation routing returns endpoints based on resolver location, not health. North American users would keep getting the failed us-east-1 ALB after the failure.

</details>

**Question 2 (Domain 2, DR strategy mapped to routing policy):**

A workload's DR plan is "pilot light": a minimal copy of the production stack runs in a secondary region but does not serve traffic until promoted. The promotion is automated via a Lambda triggered by a CloudWatch alarm on the primary's health check. The DR plan needs Route 53 to keep sending traffic to the primary until the secondary is promoted, then flip to the secondary. Which routing policy and configuration fits?

A. Failover routing with PRIMARY pointing at the primary's ALB, SECONDARY pointing at the secondary's ALB, and a Route 53 health check on the primary that triggers the failover.

B. Latency routing between the two regions; Route 53 will pick the lower-latency endpoint automatically.

C. Weighted routing with both endpoints at weight 50; the application handles regional preference at the client side.

D. Multivalue answer routing returning both endpoints; the client picks one randomly.

<details><summary>Show answer</summary>

**Correct: A.**

- A is correct: failover routing with health checks is the canonical pilot-light DR routing policy. While the primary is healthy, Route 53 returns the primary endpoint exclusively; on primary health-check failure, Route 53 starts returning the secondary endpoint. The Lambda that promotes the pilot-light stack runs in parallel; once it has promoted the stack, the secondary endpoint starts taking real traffic.
- B is wrong: latency routing does not respect a primary/secondary preference. It returns whichever endpoint is faster from the resolver's location, which can split traffic to the pilot-light region during normal operation.
- C is wrong: weighted routing at 50/50 sends half the traffic to the pilot-light region during normal operation, which defeats the pilot-light pattern.
- D is wrong: multivalue answer returns multiple endpoints and lets the client pick. The primary preference is not encoded; clients will distribute load randomly.

</details>